# Mini AT-Q — reproduce the paper's numbers and figures

This notebook regenerates the paper's tables and figures **from the released
frozen-decode data** in `data/` — no GPU or training required.

Run it from the repository root. It expects `requirements.txt` to be installed
(`numpy`, `scipy`, `matplotlib`). The tail-statistic step also uses the
`third_party/circuit-to-tensor` submodule, so make sure submodules are initialized:

```bash
git submodule update --init --recursive
pip install -r requirements.txt
```

## 1. Stage the evaluation data

`restore.sh` copies `data/results_*` to the repository root, where the
analysis scripts expect to find them.

In [ ]:
import os
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
print('repo root:', os.getcwd())
!bash data/restore.sh

## 2. Regenerate the numbers

Each script reads the CSVs and re-emits a tracked `outputs/*.json`. The regenerated
values should match the committed ones (random components are seeded).

In [ ]:
!python tools/paper_analysis.py
!python tools/paper_generality.py
!python tools/paper_hardening_numbers.py

## 3. The heavy-tail predictor (Contribution 1) and its figure

`paper_tail_statistic.py` runs 20,000 random-policy rollouts per target in a
numpy environment replica (cross-validated against the JAX environment), computes
the training-free tail statistic, and writes `outputs/figures/fig_tail_statistic.pdf`.

In [ ]:
!python tools/paper_tail_statistic.py

## 4. The mechanism, loss, and compute-grid figures

In [ ]:
!python tools/paper_mechanism_figure.py
!python tools/paper_loss_illustration.py
!python tools/paper_grid_figures.py --grid_eval results_compute_grid_20260627/eval
!python tools/paper_grid_heatmap.py

## 5. Inspect the key results

In [ ]:
import json
for name in ['generality_numbers', 'hardening_numbers', 'tail_statistic']:
    path = f'outputs/{name}.json'
    if os.path.exists(path):
        print('=' * 8, name, '=' * 8)
        print(json.dumps(json.load(open(path)), indent=2)[:1500])
        print()

In [ ]:
# Show the regenerated tail-statistic figure (Fig. 4), if a PDF/PNG viewer is available.
from IPython.display import IFrame, display
fig = 'outputs/figures/fig_tail_statistic.pdf'
display(IFrame(fig, width=560, height=520)) if os.path.exists(fig) else print('run cell 3 first')